In [ ]:
# Author: Niko Bleidistel
# last change: 2026-08-11

# Package Import

In [ ]:
from pathlib import Path 
from os import makedirs
import sys
import importlib

import mph

In [ ]:
PYTHON_HELPER_FOLDER = Path(r"py-helpers")

# Add the path to the custom packages to sys.path so that they can be imported
sys.path.append(str(PYTHON_HELPER_FOLDER.resolve()))

# import custom packages
import comsol_data_export as cde
import time_logging as tl

# reload custom packages (for each execution) to reflect recent changes
_ = importlib.reload(cde)
_ = importlib.reload(tl)

# PATHS

In [ ]:
# INPUT_FOLDER = Path(r"../MODELLE")
# OUTPUT_FOLDER = Path(r"../RESULTS")

# makedirs(OUTPUT_FOLDER, exist_ok=True)  # create output folder if it doesn't exist

In [ ]:
MAIN_FOLDER = Path(r"R:\Bleidistel_Niko\COMSOL\COMSOL Files\12_bachelor_thesis_models_low_res")
INPUT_FOLDER = MAIN_FOLDER #/ "Solved model versions"
OUTPUT_FOLDER = MAIN_FOLDER / "Test Output"

makedirs(OUTPUT_FOLDER, exist_ok=True)  # create output folder if it doesn't exist

In [ ]:
if True:
    input_folder = INPUT_FOLDER

    mph_files = list(input_folder.glob("*.mph"))

    print("Available model files in the input folder:")
    for modelfile in mph_files:
            print(modelfile.stem)

# INITIALIZE

In [ ]:
# initialize time logging
_ = tl.initialize_time_log(OUTPUT_FOLDER / 'time_log.csv')

# initialize COMSOL client (server)
client = mph.start()

# SIMULATE

## Constants

In [ ]:
EXPORT_DICT = {
    "mf.normB": "Magnetic flux density, norm [T]",
    "mf.Bx": "Magnetic flux density, x-component [T]", 
    "mf.By": "Magnetic flux density, y-component [T]", 
    "mf.Bz": "Magnetic flux density, z-component [T]",
    "T": "Temperature [K]",
}
EXPORT_PARAMS = list(EXPORT_DICT.keys())
EXPORT_DESCRIPTION = list(EXPORT_DICT.values())

CONDUCTOR_EXPORT_DICT = {
    "V": "Electric potential [V]",
    "ec.normJ": "Current density, norm [A/m^2]",
    "ec.Jx": "Current density, x-component [A/m^2]",
    "ec.Jy": "Current density, y-component [A/m^2]",
    "ec.Jz": "Current density, z-component [A/m^2]",
}
CONDUCTOR_EXPORT_PARAMS = list(CONDUCTOR_EXPORT_DICT.keys())
CONDUCTOR_EXPORT_DESCRIPTION = list(CONDUCTOR_EXPORT_DICT.values())

## Simulation Order

In [ ]:
GROUP_1 = [
    "01_01_a-Round spiral",
    "01_01_b-Rectangular spiral",
    "01_02_a-Grid",
    "01_03_a-Round spiral combined with grid",
    "01_03_b-Rectangular spiral combined with grid",
]

GROUP_2 = [
    "01_00_d-high resolution planes and increasing areas",
    "01_00_c-high resolution planes",
    "01_00_b-high resolution cuboid",
    "01_00_a-auto mesh"
]

GROUP_3 = [
    "02_00_a-H design with minimized insulator",
    "02_00_b-H design with full insulator",
]

## Simulate

### Group 1: Coil Grid Designs

In [ ]:
if True:
    input_folder = INPUT_FOLDER
    output_folder = OUTPUT_FOLDER / "01_xx-Coils and Grid Designs"

    for g1 in GROUP_1:
        modelfile = input_folder / f"{g1}.mph"

        # create output folder for the model
        model_output_folder = output_folder / modelfile.stem
        makedirs(model_output_folder, exist_ok=True)  # create output folder if it doesn't exist

        try:
            cde.simulate_model(
                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = model_output_folder,

                    # simulation settings
                    client = client,

                    export_params = EXPORT_PARAMS.copy(),
                    export_descriptions = EXPORT_DESCRIPTION.copy(),

                    conductor_export_params = CONDUCTOR_EXPORT_PARAMS.copy(),
                    conductor_export_descriptions = CONDUCTOR_EXPORT_DESCRIPTION.copy(),

                    # COMSOL internal interpolation
                    Depth_point1 = (0.0, 0.0, 0.0),
                    Depth_point2 = (0.0, 0.0, "-1*epilayer_height"),

                    Homogeneity_point1 = (0.0, "+0.5*conductor_all_length*1.2", 0.0),
                    Homogeneity_point2 = (0.0, "-0.5*conductor_all_length*1.2", 0.0),
                    Homogeneity_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Homogeneity_orth_vector = [0, 0, 1],

                    Longitudinal_point1 = ("+0.5*conductor_all_length*1.2", 0.0, 0.0),
                    Longitudinal_point2 = ("-0.5*conductor_all_length*1.2", 0.0, 0.0),
                    Longitudinal_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Longitudinal_orth_vector = [0, 0, 1],

                    xy_plane_coordinate = 0.0,
                    conductor_plane_coordinate = "0.5*conductor_all_height",

                    # boolean flags    
                    export_parameters_to_csv = True,
                    extend_export_from_params_in_csv= False,
                    show_model_info = False,
                    solve_model = True,
                    save_solved_model = True,
                    evaluate_parameter_expressions = True,
                    export_all_solution_data = False, # file size is pretty large
                    export_line_solution_data = True,
                    export_plane_solution_data = True,
                    save_small_model_version = True,
                    new_log_file = True,

                    # for sweeps
                    iteration_number = None,
                    model = None,
            )
    
        except Exception as e:
            with open(model_output_folder / 'errormessage.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")

In [ ]:
client.clear()

### Group 2: Meshing and POC

In [ ]:
if True:
    input_folder = INPUT_FOLDER
    output_folder = OUTPUT_FOLDER / "01_00-Meshing and POC"

    for g2 in GROUP_2:
        modelfile = input_folder / f"{g2}.mph"

        # create output folder for the model
        model_output_folder = output_folder / modelfile.stem
        makedirs(model_output_folder, exist_ok=True)  # create output folder if it doesn't exist

        try:
            cde.simulate_model(
                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = model_output_folder,

                    # simulation settings
                    client = client,

                    export_params = EXPORT_PARAMS.copy(),
                    export_descriptions = EXPORT_DESCRIPTION.copy(),

                    conductor_export_params = CONDUCTOR_EXPORT_PARAMS.copy(),
                    conductor_export_descriptions = CONDUCTOR_EXPORT_DESCRIPTION.copy(),

                    # COMSOL internal interpolation
                    Depth_point1 = (0.0, 0.0, "-1*insulator_height"),
                    Depth_point2 = (0.0, 0.0, "-1*epilayer_height-1*insulator_height"),

                    Homogeneity_point1 = (0.0, "+0.5*epilayer_width", "-1*insulator_height"),
                    Homogeneity_point2 = (0.0, "-0.5*epilayer_width", "-1*insulator_height"),
                    Homogeneity_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Homogeneity_orth_vector = [0, 0, 1],

                    Longitudinal_point1 = ("+0.5*epilayer_length", 0.0, "-1*insulator_height"),
                    Longitudinal_point2 = ("-0.5*epilayer_length", 0.0, "-1*insulator_height"),
                    Longitudinal_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Longitudinal_orth_vector = [0, 0, 1],

                    xy_plane_coordinate = 0.0,
                    conductor_plane_coordinate = "0.5*conductor_all_height",

                    # boolean flags    
                    export_parameters_to_csv = True,
                    extend_export_from_params_in_csv= False,
                    show_model_info = False,
                    solve_model = True,
                    save_solved_model = True,
                    evaluate_parameter_expressions = True,
                    export_all_solution_data = False, # file size is pretty large
                    export_line_solution_data = True,
                    export_plane_solution_data = True,
                    save_small_model_version = True,
                    new_log_file = True,

                    # for sweeps
                    iteration_number = None,
                    model = None,
            )
    
        except Exception as e:
            with open(model_output_folder / 'errormessage.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")

In [ ]:
client.clear()

### Group 3: H Designs

In [ ]:
if True:
    input_folder = INPUT_FOLDER
    output_folder = OUTPUT_FOLDER / "02_00-H Designs"

    for g3 in GROUP_3:
        modelfile = input_folder / f"{g3}.mph"

        # create output folder for the model
        model_output_folder = output_folder / modelfile.stem
        makedirs(model_output_folder, exist_ok=True)  # create output folder if it doesn't exist

        try:
            cde.simulate_model(
                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = model_output_folder,

                    # simulation settings
                    client = client,

                    export_params = EXPORT_PARAMS.copy(),
                    export_descriptions = EXPORT_DESCRIPTION.copy(),

                    conductor_export_params = CONDUCTOR_EXPORT_PARAMS.copy(),
                    conductor_export_descriptions = CONDUCTOR_EXPORT_DESCRIPTION.copy(),

                    # COMSOL internal interpolation
                    Depth_point1 = (0.0, 0.0, "-1*insulator_height"),
                    Depth_point2 = (0.0, 0.0, "-1*epilayer_height-1*insulator_height"),

                    Homogeneity_point1 = (0.0, "+0.5*epilayer_width", "-1*insulator_height"),
                    Homogeneity_point2 = (0.0, "-0.5*epilayer_width", "-1*insulator_height"),
                    Homogeneity_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Homogeneity_orth_vector = [0, 0, 1],

                    Longitudinal_point1 = ("+0.5*epilayer_length", 0.0, "-1*insulator_height"),
                    Longitudinal_point2 = ("-0.5*epilayer_length", 0.0, "-1*insulator_height"),
                    Longitudinal_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Longitudinal_orth_vector = [0, 0, 1],

                    xy_plane_coordinate = "-1*epilayer_height",
                    conductor_plane_coordinate = "0.5*conductor_A_height",

                    # boolean flags    
                    export_parameters_to_csv = True,
                    extend_export_from_params_in_csv= False,
                    show_model_info = False,
                    solve_model = True,
                    save_solved_model = True,
                    evaluate_parameter_expressions = True,
                    export_all_solution_data = False, # file size is pretty large
                    export_line_solution_data = True,
                    export_plane_solution_data = True,
                    save_small_model_version = True,
                    new_log_file = True,

                    # for sweeps
                    iteration_number = None,
                    model = None,
            )
    
        except Exception as e:
            with open(model_output_folder / 'errormessage.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")

In [ ]:
client.clear()

# SWEEPS
- H design: 4 strom konfis
- Round Spiral: number turns = 15, 25, 35
- Spirals + Grid: Voltage Distributions = 0, 45, 67, 90 degree (left out lines = 0) 
- Grid: Left out Lines = 0, 1, 2
    - in 3-degree increments from 0 to 90 degrees for each 'left out lines'

## Group 3: H Designs

In [ ]:
SWEEP_MODELS_3_1 = [
    "02_00_a-H design with minimized insulator",
    "02_00_b-H design with full insulator",
]

SWEEP_PRAMS_3_1 = [
    "I_conductor_A_terminal",
    "I_conductor_B_minus_terminal",
    "I_conductor_B_plus_terminal",
    ]
SWEEP_VALUES_3_1 = [
    [ 0e-6, 10e-6,  10e-6],
    [ 0e-6, 10e-6, -10e-6],
    [ 5e-6, 10e-6, -10e-6],
    [10e-6, 10e-6, -10e-6],
    ]


In [ ]:
if True:
    cde.sweep_iteration_test(
            sweep_parameters = SWEEP_PRAMS_3_1,
            sweep_values = SWEEP_VALUES_3_1,
        )

In [ ]:
if True:
    input_folder = INPUT_FOLDER
    output_folder = OUTPUT_FOLDER / "02_00-H Designs"

    for g3_1 in SWEEP_MODELS_3_1:
        modelfile = input_folder / f"{g3_1}.mph"

        # create output folder for the model
        model_output_folder = output_folder / modelfile.stem / "Sweep"
        makedirs(model_output_folder, exist_ok=True)  # create output folder if it doesn't exist

        try:
            cde.sweep_model(
                    # sweep settings
                    sweep_parameters = SWEEP_PRAMS_3_1,
                    sweep_values = SWEEP_VALUES_3_1,

                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = model_output_folder,

                    # simulation settings
                    client = client,

                    export_params = EXPORT_PARAMS.copy(),
                    export_descriptions = EXPORT_DESCRIPTION.copy(),

                    conductor_export_params = CONDUCTOR_EXPORT_PARAMS.copy(),
                    conductor_export_descriptions = CONDUCTOR_EXPORT_DESCRIPTION.copy(),

                    # COMSOL internal interpolation
                    Depth_point1 = (0.0, 0.0, "-1*insulator_height"),
                    Depth_point2 = (0.0, 0.0, "-1*epilayer_height-1*insulator_height"),

                    Homogeneity_point1 = (0.0, "+0.5*epilayer_width", "-1*insulator_height"),
                    Homogeneity_point2 = (0.0, "-0.5*epilayer_width", "-1*insulator_height"),
                    Homogeneity_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Homogeneity_orth_vector = [0, 0, 1],

                    Longitudinal_point1 = ("+0.5*epilayer_length", 0.0, "-1*insulator_height"),
                    Longitudinal_point2 = ("-0.5*epilayer_length", 0.0, "-1*insulator_height"),
                    Longitudinal_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Longitudinal_orth_vector = [0, 0, 1],

                    xy_plane_coordinate = "-1*epilayer_height",
                    conductor_plane_coordinate = "0.5*conductor_A_height",

                    # boolean flags    
                    export_parameters_to_csv = True,
                    extend_export_from_params_in_csv= False,
                    show_model_info = False,
                    solve_model = True,
                    save_solved_model = True,
                    evaluate_parameter_expressions = True,
                    export_all_solution_data = False, # file size is pretty large
                    export_line_solution_data = True,
                    export_plane_solution_data = True,
                    save_small_model_version = True,
                    new_log_file = True,
            )
    
        except Exception as e:
            with open(model_output_folder / 'errormessage.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")

In [ ]:
client.clear()

## Group 1: Coils and Grid Designs

In [ ]:
SWEEP_MODELS_1_1 = [
    "01_01_a-Round spiral",
]
SWEEP_MODELS_1_3 = [
    "01_03_a-Round spiral combined with grid",
    "01_03_b-Rectangular spiral combined with grid",
]
SWEEP_MODELS_1_2 = [
    "01_02_a-Grid",
]

### Group 1.1: Number of turns of round spiral

In [ ]:
SWEEP_PRAMS_1_1 = [
    "N_spiral_turns",
    ]

SWEEP_VALUES_1_1 = [
    [15],
    [25],
    [35],
    ]

In [ ]:
if True:
    cde.sweep_iteration_test(
            sweep_parameters = SWEEP_PRAMS_1_1,
            sweep_values = SWEEP_VALUES_1_1,
        )

In [ ]:
if True:
    input_folder = INPUT_FOLDER
    output_folder = OUTPUT_FOLDER / "01_xx-Coils and Grid Designs"

    for g1_1 in SWEEP_MODELS_1_1:
        modelfile = input_folder / f"{g1_1}.mph"

        # create output folder for the model
        model_output_folder = output_folder / modelfile.stem / "Sweep - N_spiral_turns"
        makedirs(model_output_folder, exist_ok=True)  # create output folder if it doesn't exist

        try:
            cde.sweep_model(
                    # sweep settings
                    sweep_parameters = SWEEP_PRAMS_1_1,
                    sweep_values = SWEEP_VALUES_1_1,

                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = model_output_folder,

                    # simulation settings
                    client = client,

                    export_params = EXPORT_PARAMS.copy(),
                    export_descriptions = EXPORT_DESCRIPTION.copy(),

                    conductor_export_params = CONDUCTOR_EXPORT_PARAMS.copy(),
                    conductor_export_descriptions = CONDUCTOR_EXPORT_DESCRIPTION.copy(),

                    # COMSOL internal interpolation
                    Depth_point1 = (0.0, 0.0, 0.0),
                    Depth_point2 = (0.0, 0.0, "-1*epilayer_height"),

                    Homogeneity_point1 = (0.0, "+0.5*conductor_all_length*1.2", 0.0),
                    Homogeneity_point2 = (0.0, "-0.5*conductor_all_length*1.2", 0.0),
                    Homogeneity_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Homogeneity_orth_vector = [0, 0, 1],

                    Longitudinal_point1 = ("+0.5*conductor_all_length*1.2", 0.0, 0.0),
                    Longitudinal_point2 = ("-0.5*conductor_all_length*1.2", 0.0, 0.0),
                    Longitudinal_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Longitudinal_orth_vector = [0, 0, 1],

                    xy_plane_coordinate = 0.0,
                    conductor_plane_coordinate = "0.5*conductor_all_height",

                    # boolean flags    
                    export_parameters_to_csv = True,
                    extend_export_from_params_in_csv= False,
                    show_model_info = False,
                    solve_model = True,
                    save_solved_model = True,
                    evaluate_parameter_expressions = True,
                    export_all_solution_data = False, # file size is pretty large
                    export_line_solution_data = True,
                    export_plane_solution_data = True,
                    save_small_model_version = True,
                    new_log_file = True,
            )
    
        except Exception as e:
            with open(model_output_folder / 'errormessage.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")

In [ ]:
client.clear()

### Group 1.3: Sweep: Voltage angles on combinations

In [ ]:
SWEEP_PRAMS_1_3, SWEEP_VALUES_1_3 = cde.get_voltage_sweep_dict(
    angles = [0, 45, 67, 90],
    magnitude = 10e-6,
    conductor_grid_length = 60e-6,
)

In [ ]:
if True:
    cde.sweep_iteration_test(
            sweep_parameters = SWEEP_PRAMS_1_3,
            sweep_values = SWEEP_VALUES_1_3,
        )

In [ ]:
if True:
    input_folder = INPUT_FOLDER
    output_folder = OUTPUT_FOLDER / "01_xx-Coils and Grid Designs"

    for g1_3 in SWEEP_MODELS_1_3:
        modelfile = input_folder / f"{g1_3}.mph"

        # create output folder for the model
        model_output_folder = output_folder / modelfile.stem / "Sweep - Voltage angles"
        makedirs(model_output_folder, exist_ok=True)  # create output folder if it doesn't exist

        try:
            cde.sweep_model(
                    # sweep settings
                    sweep_parameters = SWEEP_PRAMS_1_3,
                    sweep_values = SWEEP_VALUES_1_3,

                    # path settings
                    filename = modelfile.stem,
                    input_folder = modelfile.parent,
                    output_folder = model_output_folder,

                    # simulation settings
                    client = client,

                    export_params = EXPORT_PARAMS.copy(),
                    export_descriptions = EXPORT_DESCRIPTION.copy(),

                    conductor_export_params = CONDUCTOR_EXPORT_PARAMS.copy(),
                    conductor_export_descriptions = CONDUCTOR_EXPORT_DESCRIPTION.copy(),

                    # COMSOL internal interpolation
                    Depth_point1 = (0.0, 0.0, 0.0),
                    Depth_point2 = (0.0, 0.0, "-1*epilayer_height"),

                    Homogeneity_point1 = (0.0, "+0.5*conductor_all_length*1.2", 0.0),
                    Homogeneity_point2 = (0.0, "-0.5*conductor_all_length*1.2", 0.0),
                    Homogeneity_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Homogeneity_orth_vector = [0, 0, 1],

                    Longitudinal_point1 = ("+0.5*conductor_all_length*1.2", 0.0, 0.0),
                    Longitudinal_point2 = ("-0.5*conductor_all_length*1.2", 0.0, 0.0),
                    Longitudinal_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                    Longitudinal_orth_vector = [0, 0, 1],

                    xy_plane_coordinate = 0.0,
                    conductor_plane_coordinate = "0.5*conductor_all_height",

                    # boolean flags    
                    export_parameters_to_csv = True,
                    extend_export_from_params_in_csv= False,
                    show_model_info = False,
                    solve_model = True,
                    save_solved_model = True,
                    evaluate_parameter_expressions = True,
                    export_all_solution_data = False, # file size is pretty large
                    export_line_solution_data = True,
                    export_plane_solution_data = True,
                    save_small_model_version = True,
                    new_log_file = True,
            )
    
        except Exception as e:
            with open(model_output_folder / 'errormessage.txt', 'a') as f:
                f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
            print(f"Error occurred while processing {modelfile.name}")

In [ ]:
client.clear()

### Group 1.2: Angle Error and Number of grid

In [ ]:
angles = list(range(0, 91, 3))  # angles from 0 to 90 degrees in steps of 3 degrees
sweep_params_1_2, sweep_values_1_2 = cde.get_voltage_sweep_dict(
    angles = angles,
    magnitude = 10e-6,
    conductor_grid_length = 60e-6,
) # dictionary: for each terminal, a list of the corresponding voltage values for each angle in the sweep

SWEEP_PRAMS_1_2 = ["left_out_lines"] + sweep_params_1_2 # list of parameters for the sweep, including the left_out_lines parameter and the voltage parameters for each terminal
left_out_lines_values = [0, 1, 2] # list of values for the left_out_lines parameter

SWEEP_VALUES_1_2 = []
for left_out_lines in left_out_lines_values:
    left_out_lines_iteration_values = []
    for iteration in range(len(sweep_values_1_2)):
        iteration_values = [left_out_lines] +  sweep_values_1_2[iteration]
        left_out_lines_iteration_values.append(iteration_values)
    SWEEP_VALUES_1_2.append(left_out_lines_iteration_values) # for each value of left_out_lines, create a list of values for the sweep, including the left_out_lines value and the voltage values for each terminal
    

In [ ]:
if True:
    print(f"left_out_lines_values: {left_out_lines_values}")
    for idx, left_out_lines in enumerate(left_out_lines_values):
        print()
        print(f"Running sweep iteration test for left_out_lines = {left_out_lines}")
        cde.sweep_iteration_test(
                sweep_parameters = SWEEP_PRAMS_1_2,
                sweep_values = SWEEP_VALUES_1_2[idx],
            )

In [ ]:
if True:
    input_folder = INPUT_FOLDER
    output_folder = OUTPUT_FOLDER / "01_xx-Coils and Grid Designs"

    for g1_2 in SWEEP_MODELS_1_2:
        modelfile = input_folder / f"{g1_2}.mph"
        
        # create output folder for the model
        sweep_folder = output_folder / modelfile.stem / "Sweep - Voltage angles and left_out_lines"

        for idx, left_out_lines in enumerate(left_out_lines_values):
            model_output_folder = sweep_folder / f"left_out_lines - {left_out_lines}"
            makedirs(model_output_folder, exist_ok=True)  # create output folder if it doesn't exist
            try:
                cde.sweep_model(
                        # sweep settings
                        sweep_parameters = SWEEP_PRAMS_1_2,
                        sweep_values = SWEEP_VALUES_1_2[idx],

                        # path settings
                        filename = modelfile.stem,
                        input_folder = modelfile.parent,
                        output_folder = model_output_folder,

                        # simulation settings
                        client = client,

                        export_params = EXPORT_PARAMS.copy(),
                        export_descriptions = EXPORT_DESCRIPTION.copy(),

                        conductor_export_params = CONDUCTOR_EXPORT_PARAMS.copy(),
                        conductor_export_descriptions = CONDUCTOR_EXPORT_DESCRIPTION.copy(),

                        # COMSOL internal interpolation
                        Depth_point1 = (0.0, 0.0, 0.0),
                        Depth_point2 = (0.0, 0.0, "-1*epilayer_height"),

                        Homogeneity_point1 = (0.0, "+0.5*conductor_all_length*1.2", 0.0),
                        Homogeneity_point2 = (0.0, "-0.5*conductor_all_length*1.2", 0.0),
                        Homogeneity_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                        Homogeneity_orth_vector = [0, 0, 1],

                        Longitudinal_point1 = ("+0.5*conductor_all_length*1.2", 0.0, 0.0),
                        Longitudinal_point2 = ("-0.5*conductor_all_length*1.2", 0.0, 0.0),
                        Longitudinal_distances = "range((-1*epilayer_height-0)/10, (-1*epilayer_height-0)/10, -1*epilayer_height)",
                        Longitudinal_orth_vector = [0, 0, 1],

                        xy_plane_coordinate = 0.0,
                        conductor_plane_coordinate = "0.5*conductor_all_height",

                        # boolean flags    
                        export_parameters_to_csv = True,
                        extend_export_from_params_in_csv= False,
                        show_model_info = False,
                        solve_model = True,
                        save_solved_model = True,
                        evaluate_parameter_expressions = True,
                        export_all_solution_data = False, # file size is pretty large
                        export_line_solution_data = True,
                        export_plane_solution_data = True,
                        save_small_model_version = True,
                        new_log_file = True,
                )

            except Exception as e:
                with open(model_output_folder / 'errormessage.txt', 'a') as f:
                    f.write(f"Error occurred while processing {modelfile.name}: \n{str(e)}\n\n\n")
                print(f"Error occurred while processing {modelfile.name}")

In [ ]:
client.clear()

# END

In [ ]:
tl.log_message("Reached the end of the script.")